#RS Datacatlog Uploader

Data attributes DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537155636

Interaction events DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537352234

In [ ]:
import dotenv
import math
from typing import List, Dict
import json
dotenv.load_dotenv()

import os
import requests
import pandas as pd
import re

RS_API_KEY = os.getenv("RUDDERSTACK_API_KEY")
RS_API_URL = "https://api.rudderstack.com"

headers = {
    "Authorization": f"Bearer {RS_API_KEY}",
    "Content-Type": "application/json",
}

data_attributes_filepath = "data/Data attributes.csv" # path to data attributes file exported from confluence
interaction_events_filepath = "data/Interaction events.csv" # path to interaction events file exported from confluence
required_events_filepath = "data/Required_properties.csv" # path to file holding list of required properties for events
tracking_plan_name = "MAC website tracking" # Set this for tracking plan name
category_name = "MAC website"  # Set this for category name put on created events

# global vars
category_id = "" # Leave blank
df_existing_properties = pd.DataFrame()
df_existing_events = pd.DataFrame()


In [2]:
# functions
def save_event_id(event_name:str, event_id:str) -> bool:
    """
    Update event in df_event_index with passed event id
    
    params:
      event_name: name of event to update
      param event_id: event_id
      
    returns:
      True if event updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if event_name in df_event_index['event'].values:
        df_event_index.loc[df_event_index['event'] == event_name, 'event_id'] = event_id
        rc = True
    
    return rc

def save_property_id(property_name:str, property_type:str, property_id:str) -> bool:
    """
    Update all events in df_event_index that have this property with the passed property id.\n 
    
    params:
      property_name: name of property to update
      property_type: type of property to update
      property_id: id of property\n

    returns:
      True if properties updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if not(df_event_index[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type)]).empty:
        df_event_index.loc[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type), 'property_id'] = property_id
        rc = True
    
    return rc

def get_property_id(name:str) -> str|None:
    """
    Find existing property ID using passed property name.
    
    params:
      property_id: id of property to find
      
    returns:
      property name if found, otherwise None
    """
    rc = None
    # df_existing_properties 
    if not(df_existing_properties.empty):
      if name in df_existing_properties['name'].values: 
        rc = df_existing_properties.loc[df_existing_properties['name'] == name, 'id'].values[0]
    
    return rc
  
def get_event_id(name:str) -> str|None:
    """
    Find existing event ID using passed event name.
    
    params:
      event_id: id of event to find
      
    returns:
      event name if found, otherwise None
    """
    rc = None
    
    # df_exsiting_events
    if not(df_existing_events.empty):
      if name in df_existing_events['name'].values:
        rc = df_existing_events.loc[df_existing_events['name'] == name, 'id'].values[0]
    
    return rc
  
def get_existing_properties():
    """
    Get existing properties from data catalog.  Update df_existing_properties dataframe with results.
    """
    global df_existing_properties
    df_existing_properties = pd.DataFrame() # clear dataframe
    
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/properties?page=1&orderBy=name:asc", headers=headers )
    #print(json.dumps(response.json(), indent=4, sort_keys=True))

    properties = []
    if response.status_code == 200:
        properties = response.json()['data']

        # Calculate total pages    
        total_rows = response.json()['total']
        page_size = response.json()['pageSize']
        total_pages = math.ceil(total_rows / page_size)

        # Retrieve any remaining pages
        if (total_pages > 1):
            for page in range(2, total_pages + 1):
                response = requests.get(f"{RS_API_URL}/v2/catalog/properties?page={page}&orderBy=name:asc", headers=headers )
                
                if response.status_code == 200:
                    properties.extend(response.json()['data'])
    else:
        print(f"Error: [{response.status_code}] {response.json()['error']}")

    df_existing_properties = pd.DataFrame(properties)
    if df_existing_properties.empty:
        print("No existing properties found")
    else:
        print(f"Found {len(df_existing_properties)} existing properties")
        
def get_existing_events():
    """
    Get existing events from data catalog.  Update df_existing_events dataframe with results.
    """
    global df_existing_events
    df_existing_events = pd.DataFrame() # clear dataframe
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/events?page=1&orderBy=name:asc", headers=headers )
    #print(json.dumps(response.json(), indent=4, sort_keys=True))

    events = []
    if response.status_code == 200:
        events = response.json()['data']

        # Calculate total pages    
        total_rows = response.json()['total']
        page_size = response.json()['pageSize']
        total_pages = math.ceil(total_rows / page_size)

        # Retrieve any remaining pages
        if (total_pages > 1):
            for page in range(2, total_pages + 1):
                response = requests.get(f"{RS_API_URL}/v2/catalog/events?page={page}&orderBy=name:asc", headers=headers )
                
                if response.status_code == 200:
                    events.extend(response.json()['data'])
    else:
        print(f"Error: [{response.status_code}] {response.json()['error']}")
        

    df_existing_events = pd.DataFrame(events)
    if df_existing_events.empty:
        print("No existing events found")
    else:
        print(f"Found {len(df_existing_events)} existing events")

In [3]:
# Retrieve existing properties and events from data catalog
get_existing_properties() 
get_existing_events()

Found 8 existing properties
No existing events found


In [ ]:
# import interaction events file
df_events = pd.read_csv(interaction_events_filepath)
df_events['Trigger'] = df_events['Trigger'].fillna('blank') # default empty trigger values
df_events['Trigger'] = df_events['Trigger'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_events['Trigger'] = df_events['Trigger'].str.strip() # remove leading/trailing spaces
print(f"Loaded {len(df_events)} interaction events")

# import data attributes file
valid_types = ['object','string','number','integer','array','boolean','null']
df_properties = pd.read_csv(data_attributes_filepath)
df_properties = df_properties[df_properties['Data attribute'].notna()] # remove rows with missing 'Data attribute' values
df_properties['Type'] = df_properties['Type'].str.lower().str.strip() # lowercase all values and remove leading/trailing whitespace 
#df_properties['Type'] = df_properties['Type'].replace(['object', 'array', 'list'], 'string') # change 'object', 'array', and 'list'and  types to 'string' 
df_properties['Description'] = df_properties['Description'].fillna('blank') # default empty description values
df_properties['Description'] = df_properties['Description'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_properties['Description'] = df_properties['Description'].str.strip() # remove leading/trailing spaces 
print(f"Loaded {len(df_properties)} data attributes")

# Check properties have valid types, otherwise remove property and log error
invalid_rows = df_properties[~df_properties['Type'].isin(valid_types)]
if len(invalid_rows) > 0:
    print(f"Found {len(invalid_rows)} rows with invalid types:")
    for idx, row in invalid_rows.iterrows():
        print(f"  Row {idx}: '{row['Data attribute']}' has invalid type '{row['Type']}'")
    df_properties = df_properties[df_properties['Type'].isin(valid_types)]


In [ ]:
# Build an event index we will use to create the tracking plan.  Set required to False as default for all properties.  
print("Building event index")
#  Process properties data, capture property type as property name/type is unique.
property_rows = []
for idx, property in df_properties.iterrows():
    event_list = property['Associated with'].split('\n')
    for event in event_list:
        property_rows.append({"event": event, "property": property['Data attribute'], "type": property['Type'], "required":False, "event_id":None, "property_id":None})
    
df_event_index = pd.DataFrame(property_rows)
df_event_index = df_event_index.sort_values(by=['event'])

# Now search event data for any events with no properties against it (e.g. eol_complete) and add them.
event_rows = []
for idx, event in df_events.iterrows():
    if pd.isna(event['Parameters']):
        event_rows.append({"event": event['Event name'], "property": None,"type":None, "required":None, "event_id":None, "property_id":None})
df_event_extras = pd.DataFrame(event_rows)

df_event_index = pd.concat([df_event_index, df_event_extras], ignore_index=True)
df_event_index = df_event_index.sort_values(by=['event']) 

print(f"Event index built with {len(df_event_index)} rows" )


In [43]:
# Use Required_properties.csv to update the event index with properties that are required on an event.
print("Trying to load required properties... ",end="")

df_required_properties = pd.DataFrame()
update_count = 0
not_found_count = 0

# If file exists, update event_index with correct required values (e.g. True = required)
try:
    df_required_properties = pd.read_csv(required_events_filepath )
    print(f"Found file")
except FileNotFoundError: 
    print(f"File '{required_events_filepath}' does not exist, skipping updating event_index")
    df_required_properties = pd.DataFrame()

if not df_required_properties.empty:
    # Update event_index with required properties in events 
    for idx, row in df_required_properties.iterrows():
        # update event property to be required
        index_event = df_event_index[(df_event_index['event'] == row['event']) & (df_event_index['property'] == row['required_property'])]
        
        if index_event.empty:
            print(f"Event: '{row['event']}', Property: '{row['required_property']}' not found")
            not_found_count += 1
        else:
            df_event_index.loc[index_event.index, 'required'] = True
            update_count += 1

    print(f"\nSummary:\n=========")
    print(f"Rows processed: {len(df_required_properties)}")
    print(f"Updated: {update_count}")
    print(f"Not found: {not_found_count}")

Trying to load required properties... Found file
Event: 'test_event', Property: 'test_property' not found

Summary:
Rows processed: 3
Updated: 2
Not found: 1


In [8]:
# Create category to tag events with
body = {
    "name": "MAC website",
    "description": "Event stream from MAC website"
}

response = requests.post(f"{RS_API_URL}/v2/catalog/categories", headers=headers, json=body )

if (response.status_code == 200):
    print(f"Category created successfully - id: {response.json()['id']}")
    category_id = response.json()['id']
elif (response.status_code == 400):
    print(f"Category already exists - [{response.status_code}] {response.json()['error']}")
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/categories", headers=headers )
    
    for category in response.json()["data"]:
        if category["name"] == body["name"]:
            category_id = category["id"]
            print("Retrieved category ID: " + category_id)
            break   
else:
    print(f"Error creating category - [{response.status_code}] {response.json()['error']}")
    
    

Category already exists - [400] Category with name MAC website already exists
Retrieved category ID: cat_2yyODDUFmTgMo3sOMSf7RSftHpc


In [9]:
# Upload properties to data catalog, if property already exists log it and skip to next record

print("Uploading properties to data catalog...")
count = 0
for idx, row in df_properties.iterrows():

    body = {
    "name": row['Data attribute'],
    "description": row['Description'],
    "type": row['Type'],
    }

    response = requests.post(f"{RS_API_URL}/v2/catalog/properties", json=body, headers=headers )

    if response.status_code == 200:
        count += 1
        # Save property id to event index
        if not(save_property_id(row['Data attribute'],row['Type'], response.json()['id'])):
            print(f"Error updating event index for property. Name: {row['Data attribute']}, Type: {row['Type']}, ID: {response.json()['id']})")
    elif response.status_code == 400:
        print(f"Error creating property - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")
        print(f"Searching for existing property...",end="")
        property_id = get_property_id(row['Data attribute'])
        if (property_id):
            print(f"Found existing property '{row['Data attribute']}' with ID: {property_id}")
            # Save property id to event index
            if not(save_property_id(row['Data attribute'],row['Type'], property_id)):
                print(f"Error updating event index for property. Name: {row['Data attribute']}, Type: {row['Type']}, ID: {property_id}")
        else:
            print(f"Error existing property not found - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")
        
    else:
        print(f"Error creating property  - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")

print(f"Created {count}/{len(df_properties)} properties")



Uploading properties to data catalog...
Error creating property - [400] Property with name name and type string already exists 'name'
Searching for existing property...Found existing property 'name' with ID: prop_2z2DN38tgZujMuHDjWR1VIxH6TK
Error creating property - [400] Property with name category and type string already exists 'category'
Searching for existing property...Found existing property 'category' with ID: prop_2z2DMzQGMGghUNrKL5aIhHyg6YQ
Error creating property - [400] Property with name parent_page and type string already exists 'parent_page'
Searching for existing property...Found existing property 'parent_page' with ID: prop_2z2DNANJUDJP5w98jxfj8eqCN0a
Error creating property - [400] Property with name referrer and type string already exists 'referrer'
Searching for existing property...Found existing property 'referrer' with ID: prop_2z2DNIk1jR1s76P4DGsHuMhTvz2
Error creating property - [400] Property with name path and type string already exists 'path'
Searching for exi

In [14]:
# Upload events to data catalog, if event already exists log it and skip to next record
print("Uploading events to data catalog...")

count = 0
body_extras = {}
if category_id != "":
        body_extras["categoryId"] = category_id

for idx, row in df_events.iterrows():

    # create event
    body = {
    "name": row['Event name'],
    "description": row['Trigger'],
    "eventType": "track",
    **body_extras
    }
   

    response = requests.post(f"{RS_API_URL}/v2/catalog/events", json=body, headers=headers )
    
    if response.status_code == 200:
        count += 1
        # save event id to event index
        if not(save_event_id(row['Event name'], response.json()['id'])):
            print(f"Error updating event index with event. Name: {row['Event name']}, ID: {response.json()['id']}")
    elif response.status_code == 400:
        print(f"Error creating event - [{response.status_code}] {response.json()['error']} '{row['Event name']}'")
        print(f"Searching for existing event...",end="")
        event_id = get_event_id(row['Event name'])
        if (event_id):
            print(f"Found existing event '{row['Event name']}' with ID: {event_id}")
            # Save event id to event index
            if not(save_event_id(row['Event name'], event_id)):
                print(f"Error updating event index for event. Name: {row['Event name']}, ID: {event_id}")
            else:
                print(f"Updated event index for event. Name: {row['Event name']}, ID: {event_id}")
        else:
            print(f"Error existing event not found - [{response.status_code}] {response.json()['error']} '{row['Event name']}'")
    else:
        print(f"Error creating event '{row['Event name']}'- [{response.status_code}] {response.json()['error']}")

print(f"Created {count}/{len(df_events)} events")

Uploading events to data catalog...
Error creating event - [400] Name must be between 1 and 65 characters long. 'page (please note this is not an 'event' but a rudderstack call that will provide pageview data)'
Searching for existing event...Error existing event not found - [400] Name must be between 1 and 65 characters long. 'page (please note this is not an 'event' but a rudderstack call that will provide pageview data)'
Created 81/82 events


In [15]:
# Create tracking plan
tracking_plan_id = None

body = {
    "name": tracking_plan_name,
    "description": "Tracking plan for MAC website"
}

response = requests.post(f"{RS_API_URL}/v2/catalog/tracking-plans", headers=headers, json=body )
if (response.status_code == 200):
    tracking_plan_id = response.json()["id"]
    print(f"Tracking plan created successfully {tracking_plan_id}")
elif (response.status_code == 400):
    print(f"Failed to create tracking plan {response.json()}")
    print("Searching for existing tracking plan...",end="")
    # See if tracking plan already exists
    response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans", headers=headers )
    if response.status_code == 200:
        tracking_plans = response.json()["trackingPlans"]
        if (len(tracking_plans) > 0):
            for tracking_plan in tracking_plans:
                if (tracking_plan["name"] == tracking_plan_name):
                    tracking_plan_id = tracking_plan["id"]
                    print(f"found plan {tracking_plan_id}")
                    break
        else:
            print(f"Error: No plan found ({tracking_plan_name})")
    else:
        print(f"Error: Request failed to get tracking plans {response.status_code}")       
else:
    print(f"Erorr - Failed to create tracking plan {response.status_code}")

print(f"Tracking plan id: {tracking_plan_id}")

Tracking plan created successfully tp_2z4IgqboFBXmfStKLC8hYRneF8v
Tracking plan id: tp_2z4IgqboFBXmfStKLC8hYRneF8v


In [16]:
# Add events with properties into tracking plan

property_count = 0
event_count = 0
created_count = 0
error_count = 0
skip_count = 0

events = df_event_index['event'].unique()

print(f"Upserting events into tracking plan:")

for event in events:
    event_count += 1
    prop_list= []
    print(event)

    for _,row in df_event_index[df_event_index['event'] == event].iterrows():
        # Handle events with no properites (no property_id)
        if (row['property_id']):
            prop_json = {
                    "id": row['property_id'],
                    "required": row['required'],
                    "additionalProperties": False,
                    "metadata": {},
                "properties": []
            }
            prop_list.append(prop_json)
            property_count += 1
    
    body = { 
        "id":row['event_id'],
        "properties": prop_list
    }
    
    # Skip if no event_id found
    if (row['event_id']):
        response = requests.put(f"{RS_API_URL}/v2/catalog/tracking-plans/{tracking_plan_id}/events", headers=headers, json=body )
        
        if response.status_code == 200:
            created_count += 1
        elif response.status_code == 400:
            error_count += 1
            print(f" Error adding '{event}' - {response.status_code} {response.json()['error']} ")
        else:
            error_count += 1
            print(f" Failed to create event '{event}'  - {response.status_code} {response.json()['error']} ")
    else:
        skip_count += 1
        print(f" skipping event - Event id empty for '{event}' in df_event_index")

print("\n\nSummary\n--------")
    
print(f"Total rows: {len(df_event_index)}")
print(f"Events processed: {event_count}")
print(f"Properties processed: {property_count}")

print(f"\nEvents created: {created_count}")
print(f"Events skipped: {skip_count}")
print(f"Events failed: {error_count}")

   


Upserting events into tracking plan:
abandon_tool
ao_back
ao_complete
ao_next
ao_screen_loaded
ao_screen_nav
ao_start
bp_add_ach_complete
bp_add_ach_start
bp_add_service_back
bp_add_service_complete
bp_add_service_next
bp_add_service_start
bp_detail_interaction
bp_over_budget_attempt
bp_remove_service
bp_reset
bp_update_room_option
bp_view
compare_edit
compare_view
component_interaction
eol_complete
eol_next
eol_start
fap_interaction
fap_outlet_search
fap_provider_search
fap_wayfinder_next
fap_wayfinder_skip
fap_wayfinder_start
he_interaction
he_personalise
he_view
mar_back
mar_complete
mar_menu_stepper
mar_next
mar_start
mg_back
mg_complete
mg_delete
mg_detail_interaction
mg_next
mg_open
mg_start
mg_update_complete
mg_update_start
mg_view
mg_view_summary
my_costs_back
my_costs_complete
my_costs_next
my_costs_start
my_costs_summary_view
my_costs_update_start
nav_click
news_article_click
news_interaction
outlet_detail_interaction
outlet_interaction
outlet_view
page (please note this is 

---
# UTILITIES



## Delete a tracking plan and all its events and properties

Steps:

1) Set the id of the tracking plan to delete
2) Generate an event index
3) Delete the tracking plan and its events & events

## Step 1: 

In [17]:
# Find tracking plan id

tp_name = "MAC website tracking" # Name of tracking plan to find
tp_id = None 

response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans", headers=headers )

if response.status_code == 200:
    tracking_plans = response.json()["trackingPlans"]

    if (len(tracking_plans) > 0):
        for tracking_plan in tracking_plans:
            if (tracking_plan["name"] == tp_name):
                tp_id = tracking_plan["id"]
                break
else:
    print(f"Error: {response.status_code} {response.json()['error']}")
    
if tp_id is None:
    print(f"Could not find tracking plan called '{tp_name}'")
else:
    print(f"Tracking plan found: '{tp_name}', id: {tp_id}")



Tracking plan found: 'MAC website tracking', id: tp_2z4IgqboFBXmfStKLC8hYRneF8v


## Step 2:

In [23]:
# Build the event index.  We will use this to delete the events and properties connected to the tracking plan 


# Get all events on the tracking plan
print("Getting events on tracking plan... ", end="")
tp_event_ids = []

response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans/{tp_id}/events?page=1&orderBy=name:asc", headers=headers )

if response.status_code == 200:
    tp_event_ids = response.json()['data']

    # Calculate total pages    
    total_rows = response.json()['total']
    page_size = response.json()['pageSize']
    total_pages = math.ceil(total_rows / page_size)

    # Retrieve any remaining pages
    if (total_pages > 1):
        for page in range(2, total_pages + 1):
            response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans/{tp_id}/events?page={page}&orderBy=name:asc", headers=headers )
            
            if response.status_code == 200:
                tp_event_ids.extend(response.json()['data'])
else:
    print(f"Error: [{response.status_code}] {response.json()['error']}")

df_tp_event_ids = pd.DataFrame(tp_event_ids)
if df_tp_event_ids.empty:
    print("No events found on tracking plan")
else:
    print(f"Found {len(df_tp_event_ids)} events on tracking plan")
    

#Get the individual events
print("Getting events from tracking plan...", end="")
tp_events = []

if not df_tp_event_ids.empty:
    for _, row in df_tp_event_ids.iterrows():
        event_id = row['id']

        response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans/{tp_id}/events/{event_id}?format=properties", headers=headers )
        
        if response.status_code == 200:
            tp_events.append(response.json())
        elif response.status_code == 404:
            print(f"Event {event_id} not found")
        else:
            print(f"Error: [{response.status_code}] {response.json()['error']}")

df_tp_events = pd.DataFrame(tp_events)
if df_tp_events.empty:
    print("No events found")
else:
    print(f"Found {len(df_tp_events)} events")      


# Build event index to control our deletions

print("Building event index...", end="")
tp_event_index = []
for _, event in df_tp_events.iterrows():
    #print(event['name'])
    if event['properties']:
        for property in event['properties']:
            #print(f"  {property['name']} ({property['type']})")
            tp_event_index.append({
                'event': event['name'],
                'property': property['name'],
                'type': property['type'],
                'event_id': event['id'],
                'property_id': property['id']
            })
    else:
        tp_event_index.append({
            'event': event['name'],
            'property': None,
            'type': None,
            'event_id': event['id'],
            'property_id': None
        })
df_tp_event_index = pd.DataFrame(tp_event_index)
if df_tp_event_index.empty:
    print("No rows loaded into index")
else:
    print(f"Loaded {len(df_tp_event_index)} rows into index")  
    
    


Getting events on tracking plan... Found 81 events on tracking plan
Getting events from tracking plan...Found 81 events
Building event index...Loaded 406 rows into index


In [24]:
# id for eol_complete  ev_2z4Hw84v9HYLFXNvSqxKKOZg856

df_tp_event_index

,event,property,type,event_id,property_id
0,abandon_tool,interaction_type,string,ev_2z4HwBdRW4cHafcJWQHBxiXIk2Q,prop_2z4GdxJwdIvDHRiuNtAzTklpIei
1,abandon_tool,active_question,string,ev_2z4HwBdRW4cHafcJWQHBxiXIk2Q,prop_2z4GeimRAf8yHX0bx9e5kW35Kcv
2,abandon_tool,context,string,ev_2z4HwBdRW4cHafcJWQHBxiXIk2Q,prop_2z4GeO6QKqMefFFf4ZiHCqtdNNo
3,abandon_tool,tool,string,ev_2z4HwBdRW4cHafcJWQHBxiXIk2Q,prop_2z4GfC2g2z5yIJIRPqvLM0FuqqA
4,ao_back,previous_question,string,ev_2z4HyOBlwZCvP5gCLvsnZEQz6o1,prop_2z4GednKI8D7gYrYQbqe0E0aIMA
...,...,...,...,...,...
401,wayfinder_next,context,string,ev_2z4Hv4kCQmhTZ9yMjrESRLa3zI0,prop_2z4GeO6QKqMefFFf4ZiHCqtdNNo
402,wayfinder_next,updated,boolean,ev_2z4Hv4kCQmhTZ9yMjrESRLa3zI0,prop_2z4Gert1KTC3Zx6duFsZpSWU5iw
403,wayfinder_start,option_selected,string,ev_2z4HuzVo48TiRAK6cQ4sjngdgeO,prop_2z4GeAiAbTbSwPJbwM3ZmSp3lI7
404,wayfinder_start,attempt,integer,ev_2z4HuzVo48TiRAK6cQ4sjngdgeO,prop_2z4GeLAQG4oO1SjmaOhSkZsvTtZ


## Step 3:

In [ ]:
# Delete tracking plan
print("Deleting tracking plan...", end="")
tp_deleted = False

response = requests.delete(f"{RS_API_URL}/v2/catalog/tracking-plans/{tracking_plan_id}", headers=headers )
if response.status_code == 200: 
    print(f"Deleted tracking plan {tracking_plan_id}")
    tp_deleted = True
elif response.status_code == 404:
    print(f"Tracking plan {tracking_plan_id} not found")
else:
    print(f"Error: [{response.status_code}] {response.json()['error']}")


# Delete tracking plan events
print("Deleting tracking plan events:")
event_processed_count = 0
event_delete_count = 0
event_error_count = 0
event_not_found_count = 0

if not(df_tp_event_index.empty) & tp_deleted:
    #for _, row in df_tp_event_index.head(5).iterrows():
    tp_events = df_tp_event_index['event_id'].unique()
    #print(events)

    for event in tp_events:
        #print(event)    
        event_processed_count += 1  
        event_id = event
        response = requests.delete(f"{RS_API_URL}/v2/catalog/events/{event_id}", headers=headers )
        if response.status_code == 200: 
            event_delete_count += 1
        elif response.status_code == 400:
            event_not_found_count += 1
            print(f" Event {event_id} not found")
        else:
            event_error_count += 1
            print(f" Error: [{response.status_code}] {response.json()['error']}")
            
print("\n\nsummary:\n========\n")
print(f"Total events: {len(tp_events)}")
print(f"Processed {event_processed_count} events")
print(f"Deleted {event_delete_count} events")
print(f"Not found {event_not_found_count} events")
print(f"Error {event_error_count} events")


# Delete tracking plan properties
print("Deleting tracking plan properties:")
property_processed_count = 0
property_delete_count = 0
property_error_count = 0
property_not_found_count = 0

if not(df_tp_event_index.empty) & tp_deleted:
    tp_properties = df_tp_event_index['property_id'].unique() # get rid of duplicates
    tp_properties = [item for item in tp_properties if item is not None] # remove missing property_id's
    
    for property in tp_properties:
        property_processed_count += 1  
        property_id = property
        response = requests.delete(f"{RS_API_URL}/v2/catalog/properties/{property_id}", headers=headers)
        if response.status_code == 200: 
            property_delete_count += 1
        elif response.status_code == 400:
            property_not_found_count += 1
            print(f" Property {property_id} not found")
        else:
            property_error_count += 1
            print(f" Error: [{response.status_code}] {response.json()['error']}")
            
print("\n\nsummary:\n========\n")
print(f"Total properties: {len(tp_properties)}")
print(f"Processed: {property_processed_count}")
print(f"Deleted: {property_delete_count}")
print(f"Not found: {property_not_found_count}")
print(f"Errored: {property_error_count}")

Deleting tracking plan...Deleted tracking plan tp_2z4IgqboFBXmfStKLC8hYRneF8v


---
## Other utilities

In [ ]:
# Delete all properties in the data catalog (not attached to a tracking plan)

get_existing_properties() # Retrieve existing properties from data catalog

count = 0

for idx, row in df_existing_properties.iterrows():
    response = requests.delete(f"{RS_API_URL}/v2/catalog/properties/{row['id']}", headers=headers)
    
    if response.status_code == 200:
        count += 1
    else:
        print(f"Error deleting '{row['name']}' {row['id']}")

print(f"Deleted {count} properties")

In [ ]:
# Delete all events in the data catalog (not attached to a tracking plan)

get_existing_events() # get existing events from data catalog

count = 0

for idx, row in df_existing_events.iterrows():
    response = requests.delete(f"{RS_API_URL}/v2/catalog/events/{row['id']}", headers=headers )
    
    if response.status_code == 200:
        count += 1
    else:
        print(f"Error deleting '{row['name']}' {row['id']}")

print(f"Deleted {count} events")